# C3 · Too many features: when the training R² lies

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 40 minutes · **extended path**

## Objective

Add pure noise features one at a time to a model with 60 observations, until the features reach 59:
measure training R², adjusted R² and cross-validated R², and watch Ridge hold up where OLS collapses.

## Dataset

Synthetic: 60 observations, one true feature `x_true` (`y = 2·x_true + noise`) and 58 columns of independent Gaussian
noise, in `noise`. The cross-validation is a ready-made `KFold(5, shuffle=True, random_state=0)` (`cv`).

## Educational purpose

See quantitatively that the R² on the training set **can never decrease** when you add features, even
useless ones, and reaches 1 when the parameters reach the number of observations. And understand what exposes it: cross-validation.

## Task

1. For `k` from 0 to 58 build `X_k = [x_true, noise[:, :k]]` (k noise features). With `LinearRegression` save `r2_train[k]` (R² on the training data) and `r2_cv[k]` (mean of the cross-validated R², `scoring="r2"`).
2. Compute `r2_adj[k]`, the adjusted R² `1 − (1 − R²)(n − 1)/(n − p − 1)` with `p = k + 1` features. When `n − p − 1` is zero the index is not defined: put `np.nan` there.
3. Find `best_k_cv` (the k that maximizes `r2_cv`) and `negative_k` (the first k at which `r2_cv` becomes negative).
4. Repeat the cross-validation with `make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 20)))` and save `r2_cv_ridge[k]`.
5. Plot `r2_train`, `r2_adj`, `r2_cv` and `r2_cv_ridge` against `k`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
n = 60
x_true = rng.normal(size=n)
noise = rng.normal(size=(n, 58))
y = 2 * x_true + rng.normal(size=n)
cv = KFold(5, shuffle=True, random_state=0)
ks = np.arange(0, 59)

## Your work

In [ ]:
r2_train = np.empty(len(ks))
r2_adj = np.empty(len(ks))
r2_cv = np.empty(len(ks))
for i, k in enumerate(ks):
    X_k = np.column_stack([x_true, noise[:, :k]])
    ...

In [ ]:
best_k_cv = ...
negative_k = ...

In [ ]:
r2_cv_ridge = np.empty(len(ks))
for i, k in enumerate(ks):
    X_k = np.column_stack([x_true, noise[:, :k]])
    ...

In [ ]:
# plot: r2_train, r2_adj, r2_cv, r2_cv_ridge against k

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`LinearRegression().fit(X_k, y).score(X_k, y)` gives the training R²; `cross_val_score(LinearRegression(), X_k, y, cv=cv, scoring="r2").mean()` the cross-validated one. For the adjusted one: `p = k + 1` and the denominator `n - p - 1`.'):
    check(np.all(np.diff(r2_train) >= -1e-9), "the training R2 never decreases when you add features",
          "r2_train should be non-decreasing in k")
    check(r2_train[-1] > 0.999, "with 59 features and 60 observations the training R2 is 1",
          "with k = 58 the model interpolates the data: r2_train should be about 1")
    check(np.isnan(r2_adj[-1]) and not np.isnan(r2_adj[:-1]).any(),
          "the adjusted R2 is undefined only for k = 58", "use np.nan when n - p - 1 is zero, and only then")
    check(np.all(r2_adj[:-1] <= r2_train[:-1] + 1e-12), "the adjusted R2 never exceeds the training one",
          "r2_adj must be less than or equal to r2_train")
    check(best_k_cv <= 10, "cross-validation prefers a model with few features",
          "best_k_cv should be low (the additional features are just noise)")
    check(r2_cv.min() < -0.5, "cross-validation becomes very negative with too many features",
          "with many noise features r2_cv should drop below -0.5")
    check(negative_k == int(np.argmax(r2_cv < 0)) and negative_k < 40, "negative_k is the first k with a negative cv R2",
          "negative_k must be the first index for which r2_cv is less than zero")
    check(r2_cv_ridge[58] > r2_cv[58] + 0.5, "Ridge holds up much better than OLS with 59 features",
          "with k = 58 Ridge should have a much better cv R2 than OLS")
done()

---
The commented solution is in `solutions/C_break_the_model/C3_too_many_features_overfitting.ipynb`. Open it after you pass the check, or after you have really tried.